한글폰트

In [ ]:
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

In [ ]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

In [ ]:
df = pd.read_excel('/content/drive/My Drive/BigData/take_data.xlsx')
df = df.drop(df.columns[[0, 1]], axis=1)
df = df.drop(0)

df.rename(columns={'Unnamed: 2': '신고일',
                   'Unnamed: 3': '구정보',
                   'Unnamed: 4': '주소',
                   'Unnamed: 5': '유형',
                   'Unnamed: 6': '조치일'}, inplace=True)


강남구 데이터만 추출한 코드

In [ ]:
gangnam_data = df.loc[df['구정보'] == '강남구']

first_row = df.iloc[0]

gangnam_data = pd.concat([pd.DataFrame([first_row]), gangnam_data], ignore_index=True)

gangnam_data = gangnam_data.drop(['신고일','조치일'], axis=1)
gangnam_data =gangnam_data.drop(0)

견인이 가장 많이 된 주소 순위

In [ ]:
gangnam_dong_series = gangnam_data['주소'].str.split(' ').str[2]

grouped_data = gangnam_data.groupby(gangnam_dong_series)

for group, data in grouped_data:
    data.reset_index(drop=True, inplace=True)

In [ ]:
dong_counts = gangnam_data['주소'].value_counts()
dong_counts_ranked = dong_counts.reset_index().rename(columns={'index': '주소'})
dong_counts_ranked.index += 1
dong_counts_ranked.index.name = '순위'


dong_types = []
for address in dong_counts_ranked['주소']:
    address_data = gangnam_data[gangnam_data['주소'] == address]
    types = ', '.join(address_data['유형'].unique())
    dong_types.append(types)

dong_counts_ranked['유형'] = dong_types

dong_counts_ranked

In [ ]:
dong_counts_ranked_filtered = dong_counts_ranked[dong_counts_ranked['count'] > 1] # 견인횟수 정하는 부분 k>1 는 1회이상임, k>2면 3회 이상

dong_counts_ranked_filtered.index = range(1, len(dong_counts_ranked_filtered) + 1)

dong_counts_ranked_filtered

In [ ]:
from IPython.display import IFrame

google_map_url = "https://www.google.com/maps/embed/v1/place?key=YOUR_GOOGLE_MAPS_API_KEY&q=Space+Needle,Seattle+WA"

In [ ]:
import os
api_key = os.environ['GOOGLE_MAPS_API_KEY']  # export GOOGLE_MAPS_API_KEY=...

In [ ]:
import requests
import matplotlib.pyplot as plt

def geocode(address, api_key):
  base_url = 'https://maps.googleapis.com/maps/api/geocode/json'
  params = {
    'address': address,
    'key': api_key
  }
  response = requests.get(base_url, params=params)
  if response.status_code == 200:
    data = response.json()
    if data['status'] == 'OK':
      return data['results'][0]['geometry']['location']['lat'], data['results'][0]['geometry']['location']['lng']
    else:
      print(f"주소를 찾을 수 없습니다: {address}")
      return None, None
  else:
    print("Failed to fetch geocode data. Status code:", response.status_code)
    return None, None

addresses = dong_counts_ranked_filtered['주소'].tolist() # 입력하는 주소 데이터들

latitudes, longitudes = [], []
for address in addresses:
  latitude, longitude = geocode(address, api_key)
  if latitude is not None and longitude is not None:
    latitudes.append(latitude)
    longitudes.append(longitude)
  else:
    print(f"주소를 찾을 수 없습니다: {address}")

print(latitudes)
print(longitudes)

In [ ]:
import numpy as np

lat_data = np.array(latitudes)
lon_data = np.array(longitudes)

In [ ]:
combined_data = np.concatenate((lat_data[:, None], lon_data[:, None]), axis=1) # 2차원 데이터 변환

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs

kmeans = KMeans(n_clusters=4)
kmeans.fit(combined_data)
cluster_labels = kmeans.labels_

cluster_centers = kmeans.cluster_centers_

plt.scatter(combined_data[:, 1], combined_data[:, 0], c=cluster_labels, cmap='viridis', alpha=0.5)
plt.scatter(cluster_centers[:, 1], cluster_centers[:, 0], c='red', s=100, alpha=0.5)
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('K-Means Clustering')
plt.show()

In [ ]:
values_in_range_lon = []
for i, value in enumerate(longitudes):
    if value > 127.07: #필터
        values_in_range_lon.append((i, value))

if values_in_range_lon:
    print("Values 127.07:")
    for idx, value in values_in_range_lon:
        print(f"Index: {idx}, Value: {value}")
else:
    print("No value found in the specified range.")


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

threshold = 127.07

filtered_data = combined_data[combined_data[:, 1] <= threshold]

cluster_range = range(4, 15)
silhouette_scores = []

for num_clusters in cluster_range:
    kmeans = KMeans(n_clusters=num_clusters)
    cluster_labels = kmeans.fit_predict(filtered_data)
    silhouette_avg = silhouette_score(filtered_data, cluster_labels)
    silhouette_scores.append(silhouette_avg)

plt.plot(cluster_range, silhouette_scores, marker='o')
plt.xlabel('Number of clusters')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Analysis')
plt.show()

In [ ]:
threshold = 127.07

filtered_data = combined_data[combined_data[:, 1] <= threshold]

kmeans = KMeans(n_clusters=10)

kmeans.fit(filtered_data)
cluster_labels = kmeans.labels_
cluster_centers = kmeans.cluster_centers_

plt.scatter(filtered_data[:, 1], filtered_data[:, 0], c=cluster_labels, cmap='viridis', alpha=0.5)
plt.scatter(cluster_centers[:, 1], cluster_centers[:, 0], c='red', s=100, alpha=0.5)
for i, center in enumerate(cluster_centers):
    plt.text(center[1], center[0], f'{i+1}', fontsize=12, ha='center', va='center', color='black')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Cluster Centers')
plt.show()


In [ ]:
threshold = 127.05
threshold2 = 37.515
threshold3 = 127.03


filtered_data2 = filtered_data[(filtered_data[:, 1] <= threshold) | (filtered_data[:, 0] <= threshold2)]
filtered_data3 = filtered_data2[filtered_data2[:, 1] >= threshold3]

kmeans = KMeans(n_clusters=10)

kmeans.fit(filtered_data3)
cluster_labels = kmeans.labels_
cluster_centers = kmeans.cluster_centers_

plt.scatter(filtered_data3[:, 1], filtered_data3[:, 0], c=cluster_labels, cmap='viridis', alpha=0.5)
plt.scatter(cluster_centers[:, 1], cluster_centers[:, 0], c='red', s=100, alpha=0.5)
for i, center in enumerate(cluster_centers):
    plt.text(center[1], center[0], f'{i+1}', fontsize=12, ha='center', va='center', color='black')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Cluster Centers')
plt.show()

In [ ]:
# 클러스터링 결과 및 클러스터 중심 좌표 플로팅
cluster_labels = kmeans.labels_
cluster_centers = kmeans.cluster_centers_

plt.scatter(filtered_data3[:, 1], filtered_data3[:, 0], c=cluster_labels, cmap='viridis', alpha=0.5)
plt.scatter(cluster_centers[:, 1], cluster_centers[:, 0], c='red', s=100, alpha=0.5)

# 클러스터 중심 좌표에 번호 부여하여 표시
for i, center in enumerate(cluster_centers):
    plt.text(center[1], center[0], f'{i+1}', fontsize=12, ha='center', va='center', color='black')

plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Cluster Centers')

# 클러스터링된 데이터 개수 표시
unique, counts = np.unique(cluster_labels, return_counts=True)
cluster_counts = dict(zip(unique, counts))
for i, (cluster, count) in enumerate(cluster_counts.items()):
    plt.text(cluster_centers[i, 1], cluster_centers[i, 0], f'Cluster {cluster+1}\nCount: {count}', fontsize=10, ha='left', va='bottom', color='blue')

plt.show()

# 클러스터링된 결과의 우선순위 부여
sorted_clusters = sorted(cluster_counts.items(), key=lambda x: x[1], reverse=True)
priorities = {cluster: priority+1 for priority, (cluster, _) in enumerate(sorted_clusters)}
print("Cluster Priorities:", priorities)

# 각 클러스터의 우선순위 표시
for cluster, priority in priorities.items():
    print(f"클러스터 {cluster+1}는 {priority}등입니다.")


In [ ]:
cluster_centers = kmeans.cluster_centers_ # 중심값
print("Cluster centers:")
for i, center in enumerate(cluster_centers):
    print(f"{center[0]},{center[1]}")

In [ ]:
parking_lat = [37.51757, 37.5128028, 37.500255, 37.5185909, 37.513655, 37.5167698, 37.5169928, 37.5174488, 37.5081321, 37.5193619]
parking_long = [127.041488, 127.0537382, 127.0380871, 127.0507369, 127.0304903, 127.0200738, 127.0417958, 127.0406346, 127.0619116, 127.0503292]
# 현재 주차구역

In [ ]:
parking_lat = np.array(parking_lat)
parking_long = np.array(parking_long)

In [ ]:
parking_data = np.concatenate((parking_lat[:, None], parking_long[:, None]), axis=1) # 주차데이터 2차원 변환
combined_data = np.concatenate((lat_data[:, None], lon_data[:, None]), axis=1) # 견인데이터 2차원 변환
new_parking_data =  np.array(cluster_centers) # 새로운 주차구역

In [ ]:
# 산점도 그래프 그리기
plt.scatter(parking_data[:, 0], parking_data[:, 1], color='blue', label='Parking Data')
plt.scatter(combined_data[:, 0], combined_data[:, 1], color='red', label='Combined Data')
plt.scatter(new_parking_data[:, 0], new_parking_data[:, 1], color='green', label='New Data')

# 그래프에 레이블 추가

plt.title('Parking')
plt.legend()

# 그래프 표시
plt.show()

In [ ]:
from IPython.display import IFrame

google_map_url = "https://www.google.com/maps/embed/v1/place?key=YOUR_GOOGLE_MAPS_API_KEY&q=Space+Needle,Seattle+WA"

In [ ]:
cluster_centers = kmeans.cluster_centers_
centers_list = []
for i, center in enumerate(cluster_centers):
    center_info = [center[0], center[1]]
    centers_list.append(center_info)
print("Cluster centers as list:", centers_list)


In [ ]:
import os
api_key = os.environ['KAKAO_API_KEY']  # export KAKAO_API_KEY=...

In [ ]:
import requests

radius = 500

for center_coord in cluster_centers:
    url = f'https://dapi.kakao.com/v2/local/search/category.json?category_group_code=SW8&x={center_coord[1]}&y={center_coord[0]}&radius={radius}'
    headers = {'Authorization': f'KakaoAK {api_key}'}
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        data = response.json()
        if data['meta']['total_count'] > 0:
            print(f"클러스터 중심 좌표 {center_coord} 근처에 있는 지하철역/버스 정류장:")
            for place in data['documents']:
                print(place['place_name'], place['distance'])
        else:
            print(f"클러스터 중심 좌표 {center_coord} 근처에는 지하철역/버스 정류장이 없습니다.")
    else:
        print("API 호출에 실패했습니다.")


In [ ]:
import requests

def get_address_from_coordinates(lat, lon, api_key):
    url = f'https://dapi.kakao.com/v2/local/geo/coord2address.json?x={lon}&y={lat}'
    headers = {'Authorization': f'KakaoAK {api_key}'}
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        data = response.json()
        if data.get('documents'):
            return data['documents'][0]['address']
    return None

def extract_dong_from_address(address):
    return address.get('region_3depth_name', None)

for center_coord in cluster_centers:
    address_info = get_address_from_coordinates(center_coord[0], center_coord[1], api_key)
    if address_info:
        dong = extract_dong_from_address(address_info)
        if dong:
            print(f"클러스터 중심 좌표 {center_coord}는 '{dong}'에 속합니다.")
        else:
            print(f"클러스터 중심 좌표 {center_coord}의 동 정보를 가져올 수 없습니다.")
    else:
        print(f"클러스터 중심 좌표 {center_coord}의 주소 정보를 가져올 수 없습니다.")
